In [1]:
# ============================================================
# ScamShield AI — Notebook 6: SHAP Explainability
# ============================================================
#
# SHAP = SHapley Additive exPlanations
#
# WHAT IS SHAP?
# Imagine you and 3 friends order food together.
# The total bill is Rs 1000.
# How much did EACH person's order contribute to the bill?
# SHAP answers this for ML models:
# "How much did EACH feature (word) contribute to the prediction?"
#
# SHAP gives every feature a value:
#   Positive SHAP = pushed score TOWARD scam
#   Negative SHAP = pushed score AWAY from scam (toward legit)
#
# WHY THIS MATTERS FOR US:
# User sees "95% scam" → they want to know WHY
# SHAP tells us: "verify(+0.34), OTP(+0.28), blocked(+0.21)"
# We translate this to: "Urgency + OTP request detected"
# ============================================================

import os
import re
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# SHAP
import shap

# Sklearn
from sklearn.pipeline import Pipeline

# Deep translator for multilingual
from deep_translator import GoogleTranslator

# Lang detection
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42  # Make language detection reproducible

# NLTK
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

# Display
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-darkgrid')

print("=" * 60)
print("  ScamShield AI — SHAP Explainability Layer")
print("=" * 60)
print()

# ── Load Saved Models ─────────────────────────────────────
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'backend', 'saved_models')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

print("Loading saved models...")

# Load TF-IDF vectorizer
with open(os.path.join(MODELS_DIR, 'tfidf_vectorizer.pkl'), 'rb') as f:
    tfidf = pickle.load(f)
print("  [OK] TF-IDF Vectorizer")

# Load Logistic Regression
with open(os.path.join(MODELS_DIR, 'text_classifier_lr.pkl'), 'rb') as f:
    lr_model = pickle.load(f)
print("  [OK] Logistic Regression")

# Load XGBoost text classifier
with open(os.path.join(MODELS_DIR, 'text_classifier_xgb.pkl'), 'rb') as f:
    xgb_model = pickle.load(f)
print("  [OK] XGBoost Text Classifier")

# Load URL classifier
with open(os.path.join(MODELS_DIR, 'url_classifier_xgb.pkl'), 'rb') as f:
    url_model = pickle.load(f)
print("  [OK] URL XGBoost Classifier")

# Load URL feature info
with open(os.path.join(MODELS_DIR, 'url_feature_info.json'), 'r') as f:
    url_feature_info = json.load(f)
print("  [OK] URL Feature Info")

# Load ensemble config
with open(os.path.join(MODELS_DIR, 'ensemble_config.json'), 'r') as f:
    ensemble_config = json.load(f)
print("  [OK] Ensemble Config")

# Load test data
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'text_train.csv'))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, 'text_test.csv'))

train_df['cleaned_text'] = train_df['cleaned_text'].fillna('').astype(str)
test_df['cleaned_text']  = test_df['cleaned_text'].fillna('').astype(str)

X_train_text = train_df['cleaned_text'].values
X_test_text  = test_df['cleaned_text'].values
y_test       = test_df['label'].values.astype(int)

print()
print("All models loaded successfully!")

  ScamShield AI — SHAP Explainability Layer

Loading saved models...
  [OK] TF-IDF Vectorizer
  [OK] Logistic Regression
  [OK] XGBoost Text Classifier
  [OK] URL XGBoost Classifier
  [OK] URL Feature Info
  [OK] Ensemble Config

All models loaded successfully!


In [2]:
# ============================================================
# SHAP EXPLAINER — LOGISTIC REGRESSION
# ============================================================
# For linear models like LR, we use LinearExplainer.
# It's fast and exact (not approximated).
#
# How it works:
# LR assigns a weight to each word.
# SHAP value = weight × (TF-IDF score - background mean)
# This tells us: compared to an average message,
# how much did this specific word contribute?
# ============================================================

print("Building SHAP explainer for Logistic Regression...")
print()

# Transform training data through TF-IDF
X_train_tfidf = tfidf.transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

# Create SHAP LinearExplainer
# background = training data distribution
# SHAP compares each prediction against this background
lr_explainer = shap.LinearExplainer(
    lr_model,
    X_train_tfidf,
    feature_perturbation="interventional"
)

print("Computing SHAP values for test set...")
print("(This may take 1-2 minutes...)")

# Compute SHAP values for test samples
# We use a subset for speed (first 50 samples)
n_explain = min(50, len(X_test_text))
X_explain_tfidf = X_test_tfidf[:n_explain]

lr_shap_values = lr_explainer.shap_values(X_explain_tfidf)

print(f"SHAP values computed!")
print(f"  Shape: {lr_shap_values.shape}")
print(f"  → {lr_shap_values.shape[0]} samples × {lr_shap_values.shape[1]} features")
print()

# Get feature names from TF-IDF
feature_names = tfidf.get_feature_names_out()

print("SHAP Analysis for first test sample:")
sample_idx = 0
sample_text = X_test_text[sample_idx]
sample_label = y_test[sample_idx]
sample_shap = lr_shap_values[sample_idx]

print(f"  Text:  '{sample_text[:80]}...'")
print(f"  Label: {'SCAM' if sample_label == 1 else 'LEGITIMATE'}")
print()

# Find top contributing features for this sample
top_pos_idx = np.argsort(sample_shap)[-10:][::-1]  # Top scam indicators
top_neg_idx = np.argsort(sample_shap)[:10]          # Top legit indicators

print("  Top SCAM indicators (positive SHAP):")
for idx in top_pos_idx:
    if sample_shap[idx] > 0:
        bar = '█' * int(sample_shap[idx] * 100)
        print(f"    '{feature_names[idx]}': +{sample_shap[idx]:.4f} {bar}")

print()
print("  Top LEGITIMATE indicators (negative SHAP):")
for idx in top_neg_idx:
    if sample_shap[idx] < 0:
        bar = '█' * int(abs(sample_shap[idx]) * 100)
        print(f"    '{feature_names[idx]}': {sample_shap[idx]:.4f} {bar}")

Building SHAP explainer for Logistic Regression...

Computing SHAP values for test set...
(This may take 1-2 minutes...)
SHAP values computed!
  Shape: (50, 10000)
  → 50 samples × 10000 features

SHAP Analysis for first test sample:
  Text:  'fine simply sitting...'
  Label: LEGITIMATE

  Top SCAM indicators (positive SHAP):
    'simply': +0.2414 ████████████████████████
    'sorry': +0.0260 ██
    'still': +0.0230 ██
    'much': +0.0188 █
    'yeah': +0.0148 █
    'got': +0.0140 █
    'later': +0.0131 █
    'aight': +0.0119 █
    'lor': +0.0113 █
    'pls': +0.0113 █

  Top LEGITIMATE indicators (negative SHAP):
    'fine': -0.3783 █████████████████████████████████████
    'sitting': -0.0773 ███████
    'text': -0.0613 ██████
    'call': -0.0462 ████
    'message': -0.0210 ██
    'chat': -0.0165 █
    'new': -0.0150 █
    'url_present': -0.0146 █
    'txt': -0.0138 █
    'free': -0.0101 █


In [3]:
# ============================================================
# SHAP EXPLAINER — LOGISTIC REGRESSION
# ============================================================
# For linear models like LR, we use LinearExplainer.
# It's fast and exact (not approximated).
#
# How it works:
# LR assigns a weight to each word.
# SHAP value = weight × (TF-IDF score - background mean)
# This tells us: compared to an average message,
# how much did this specific word contribute?
# ============================================================

print("Building SHAP explainer for Logistic Regression...")
print()

# Transform training data through TF-IDF
X_train_tfidf = tfidf.transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

# Create SHAP LinearExplainer
# background = training data distribution
# SHAP compares each prediction against this background
lr_explainer = shap.LinearExplainer(
    lr_model,
    X_train_tfidf,
    feature_perturbation="interventional"
)

print("Computing SHAP values for test set...")
print("(This may take 1-2 minutes...)")

# Compute SHAP values for test samples
# We use a subset for speed (first 50 samples)
n_explain = min(50, len(X_test_text))
X_explain_tfidf = X_test_tfidf[:n_explain]

lr_shap_values = lr_explainer.shap_values(X_explain_tfidf)

print(f"SHAP values computed!")
print(f"  Shape: {lr_shap_values.shape}")
print(f"  → {lr_shap_values.shape[0]} samples × {lr_shap_values.shape[1]} features")
print()

# Get feature names from TF-IDF
feature_names = tfidf.get_feature_names_out()

print("SHAP Analysis for first test sample:")
sample_idx = 0
sample_text = X_test_text[sample_idx]
sample_label = y_test[sample_idx]
sample_shap = lr_shap_values[sample_idx]

print(f"  Text:  '{sample_text[:80]}...'")
print(f"  Label: {'SCAM' if sample_label == 1 else 'LEGITIMATE'}")
print()

# Find top contributing features for this sample
top_pos_idx = np.argsort(sample_shap)[-10:][::-1]  # Top scam indicators
top_neg_idx = np.argsort(sample_shap)[:10]          # Top legit indicators

print("  Top SCAM indicators (positive SHAP):")
for idx in top_pos_idx:
    if sample_shap[idx] > 0:
        bar = '█' * int(sample_shap[idx] * 100)
        print(f"    '{feature_names[idx]}': +{sample_shap[idx]:.4f} {bar}")

print()
print("  Top LEGITIMATE indicators (negative SHAP):")
for idx in top_neg_idx:
    if sample_shap[idx] < 0:
        bar = '█' * int(abs(sample_shap[idx]) * 100)
        print(f"    '{feature_names[idx]}': {sample_shap[idx]:.4f} {bar}")

Building SHAP explainer for Logistic Regression...

Computing SHAP values for test set...
(This may take 1-2 minutes...)
SHAP values computed!
  Shape: (50, 10000)
  → 50 samples × 10000 features

SHAP Analysis for first test sample:
  Text:  'fine simply sitting...'
  Label: LEGITIMATE

  Top SCAM indicators (positive SHAP):
    'simply': +0.2414 ████████████████████████
    'sorry': +0.0260 ██
    'still': +0.0230 ██
    'much': +0.0188 █
    'yeah': +0.0148 █
    'got': +0.0140 █
    'later': +0.0131 █
    'aight': +0.0119 █
    'lor': +0.0113 █
    'pls': +0.0113 █

  Top LEGITIMATE indicators (negative SHAP):
    'fine': -0.3783 █████████████████████████████████████
    'sitting': -0.0773 ███████
    'text': -0.0613 ██████
    'call': -0.0462 ████
    'message': -0.0210 ██
    'chat': -0.0165 █
    'new': -0.0150 █
    'url_present': -0.0146 █
    'txt': -0.0138 █
    'free': -0.0101 █


In [4]:
# ============================================================
# CORE SHAP EXPLANATION FUNCTION
# ============================================================
# This function is what our FastAPI backend will call.
# Given a text, it returns the top contributing words
# and their SHAP values.
# ============================================================

def get_text_shap_explanation(
    text,
    tfidf,
    model,
    explainer,
    feature_names,
    top_n=10
):
    """
    Get SHAP-based word-level explanation for a text prediction.
    
    Args:
        text:          Input text string to explain
        tfidf:         Fitted TF-IDF vectorizer
        model:         Trained classifier
        explainer:     SHAP explainer for the model
        feature_names: Array of feature names from TF-IDF
        top_n:         Number of top features to return
    
    Returns:
        dict: {
            'prediction': 0 or 1,
            'probability': float 0-1,
            'top_scam_words': list of (word, shap_value) tuples,
            'top_legit_words': list of (word, shap_value) tuples,
            'all_contributions': dict of word→shap_value
        }
    """
    
    # Step 1: Vectorize the input text
    text_vectorized = tfidf.transform([text])
    
    # Step 2: Get model prediction
    prediction  = model.predict(text_vectorized)[0]
    probability = model.predict_proba(text_vectorized)[0][1]
    
    # Step 3: Compute SHAP values
    shap_vals = explainer.shap_values(text_vectorized)
    shap_array = shap_vals[0]  # First (only) sample
    
    # Step 4: Get words present in this text
    # Only features with non-zero TF-IDF contribute
    text_features = text_vectorized.toarray()[0]
    present_mask  = text_features != 0
    
    present_features = feature_names[present_mask]
    present_shap     = shap_array[present_mask]
    
    # Step 5: Sort by SHAP value
    sorted_order = np.argsort(present_shap)[::-1]
    
    # Top scam words (highest positive SHAP)
    top_scam_words = []
    for idx in sorted_order:
        if present_shap[idx] > 0:
            top_scam_words.append({
                'word':       str(present_features[idx]),
                'shap_value': float(present_shap[idx]),
                'impact':     'scam'
            })
        if len(top_scam_words) >= top_n:
            break
    
    # Top legit words (most negative SHAP)
    top_legit_words = []
    for idx in sorted_order[::-1]:
        if present_shap[idx] < 0:
            top_legit_words.append({
                'word':       str(present_features[idx]),
                'shap_value': float(present_shap[idx]),
                'impact':     'legitimate'
            })
        if len(top_legit_words) >= top_n:
            break
    
    # All contributions dict
    all_contributions = {
        str(present_features[i]): float(present_shap[i])
        for i in range(len(present_features))
    }
    
    return {
        'prediction':       int(prediction),
        'probability':      float(probability),
        'top_scam_words':   top_scam_words,
        'top_legit_words':  top_legit_words,
        'all_contributions': all_contributions
    }


# ── Test the function ─────────────────────────────────────
test_messages = [
    "URGENT: Your SBI account will be blocked! Send OTP 847291 to verify immediately. Click: http://sbi-verify.xyz",
    "Your Amazon order #12345 has been shipped. Expected delivery Jan 20.",
    "Congratulations! You won Rs 50000 in KBC Lucky Draw. Call 9876543210 to claim.",
    "Meeting tomorrow at 10am. Please bring the quarterly reports."
]

print("Testing SHAP explanation function:")
print()

for msg in test_messages:
    # Clean text first (same preprocessing as training)
    result = get_text_shap_explanation(
        msg, tfidf, lr_model, lr_explainer, feature_names
    )
    
    label = "🔴 SCAM" if result['prediction'] == 1 else "✅ LEGIT"
    print(f"Text: '{msg[:70]}...'")
    print(f"Prediction: {label} ({result['probability']*100:.1f}%)")
    
    if result['top_scam_words']:
        scam_words = [f"'{w['word']}'" for w in result['top_scam_words'][:5]]
        print(f"Top scam signals: {', '.join(scam_words)}")
    
    if result['top_legit_words']:
        legit_words = [f"'{w['word']}'" for w in result['top_legit_words'][:3]]
        print(f"Legit signals: {', '.join(legit_words)}")
    print()

Testing SHAP explanation function:

Text: 'URGENT: Your SBI account will be blocked! Send OTP 847291 to verify im...'
Prediction: 🔴 SCAM (74.6%)
Top scam signals: 'verify', 'urgent', 'account', 'otp', 'immediately'
Legit signals: 'sbi account'

Text: 'Your Amazon order #12345 has been shipped. Expected delivery Jan 20....'
Prediction: ✅ LEGIT (36.4%)
Top scam signals: 'order', 'delivery', 'amazon', 'amazon order'
Legit signals: 'jan', 'shipped', 'expected'

Text: 'Congratulations! You won Rs 50000 in KBC Lucky Draw. Call 9876543210 t...'
Prediction: 🔴 SCAM (95.1%)
Top scam signals: 'claim', 'call', 'draw', 'lucky'

Text: 'Meeting tomorrow at 10am. Please bring the quarterly reports....'
Prediction: ✅ LEGIT (12.2%)
Top scam signals: 'please'
Legit signals: 'meeting', 'tomorrow', 'bring'



In [5]:
# ============================================================
# REASON GENERATOR
# ============================================================
# Converts raw SHAP values + features into human-readable
# explanations that a non-technical user can understand.
#
# APPROACH:
# We define "signal categories" — groups of words that
# indicate specific scam tactics.
# When SHAP tells us "verify" is important,
# we map it to "Verification/Urgency request detected"
# ============================================================

# ── Signal Category Mapping ───────────────────────────────
# Each category has:
#   - keywords: words that trigger this category
#   - reason: human-readable explanation
#   - severity: how dangerous this signal is
#   - advice: what to do about this specific signal

SIGNAL_CATEGORIES = {
    'urgency': {
        'keywords': [
            'urgent', 'immediately', 'now', 'asap', 'quick',
            'hurry', 'expire', 'expir', 'last chance', 'today',
            'hour', '24', '48', 'limited', 'deadline'
        ],
        'reason': 'Urgency language detected — scammers use time pressure to prevent you from thinking clearly',
        'severity': 'HIGH',
        'advice': 'Take your time. Legitimate organizations never demand immediate action via SMS/email.'
    },
    'account_threat': {
        'keywords': [
            'block', 'suspend', 'deactivat', 'terminat', 'clos',
            'freeze', 'restrict', 'limit', 'disabl', 'lock'
        ],
        'reason': 'Account threat detected — threatening to block/suspend your account to scare you',
        'severity': 'HIGH',
        'advice': 'Call your bank/service directly using the official number on their website.'
    },
    'verification_request': {
        'keywords': [
            'verify', 'verif', 'confirm', 'validat', 'authent',
            'kyc', 'update', 'complet', 'submit'
        ],
        'reason': 'Verification request detected — asking you to "verify" personal information',
        'severity': 'HIGH',
        'advice': 'Never verify personal details through links in SMS/email. Use official apps/websites.'
    },
    'otp_request': {
        'keywords': [
            'otp', 'one time', 'passcode', 'pin', 'password',
            'share otp', 'enter otp', 'send otp'
        ],
        'reason': 'OTP/Password request detected — NO legitimate service asks you to share your OTP',
        'severity': 'CRITICAL',
        'advice': 'NEVER share OTP with anyone. Banks/UPI apps will NEVER ask for your OTP.'
    },
    'prize_lottery': {
        'keywords': [
            'won', 'winner', 'prize', 'lottery', 'lucky', 'reward',
            'gift', 'congratulation', 'selected', 'chosen', 'kbc'
        ],
        'reason': 'Prize/Lottery scam detected — you cannot win a contest you never entered',
        'severity': 'HIGH',
        'advice': 'Ignore completely. Report to cybercrime.gov.in if you get such messages.'
    },
    'financial_lure': {
        'keywords': [
            'cash', 'money', 'rupee', 'rs', 'lakh', 'crore',
            'earn', 'income', 'profit', 'invest', 'return',
            'cashback', 'refund', 'free'
        ],
        'reason': 'Financial lure detected — offering money/prizes to lower your guard',
        'severity': 'MEDIUM',
        'advice': 'If it sounds too good to be true, it is. Verify through official channels only.'
    },
    'suspicious_url': {
        'keywords': [
            'url_present', 'http', 'click', 'link', 'visit',
            'website', 'portal', 'login', 'signin'
        ],
        'reason': 'Suspicious link detected — contains URL that may lead to a fake website',
        'severity': 'HIGH',
        'advice': 'Do NOT click links in SMS/email. Type the official website address manually.'
    },
    'personal_info_request': {
        'keywords': [
            'card', 'account', 'number', 'detail', 'information',
            'pan', 'aadhaar', 'dob', 'birth', 'address',
            'bank', 'ifsc', 'credit', 'debit', 'cvv'
        ],
        'reason': 'Personal information request detected — asking for sensitive data',
        'severity': 'CRITICAL',
        'advice': 'Never share card numbers, CVV, Aadhaar, or PAN through SMS/email/phone.'
    },
    'upi_fraud': {
        'keywords': [
            'upi', 'paytm', 'phonepe', 'gpay', 'google pay',
            'bhim', 'neft', 'imps', 'transfer', 'send money'
        ],
        'reason': 'UPI/Payment fraud pattern detected — targeting digital payment systems',
        'severity': 'CRITICAL',
        'advice': 'Never share UPI PIN. Use only official bank apps. Enable transaction limits.'
    },
    'impersonation': {
        'keywords': [
            'sbi', 'hdfc', 'icici', 'axis', 'rbi', 'sebi',
            'amazon', 'flipkart', 'paytm', 'jio', 'airtel',
            'government', 'police', 'court', 'income tax',
            'irctc', 'bsnl', 'uidai'
        ],
        'reason': 'Brand/Authority impersonation detected — pretending to be a trusted organization',
        'severity': 'HIGH',
        'advice': 'Contact the organization directly using their official website/app to verify.'
    }
}


def generate_reasons(
    shap_explanation,
    cleaned_text,
    risk_score,
    url_features=None
):
    """
    Generate human-readable reasons for the scam prediction.
    
    Args:
        shap_explanation: Output from get_text_shap_explanation()
        cleaned_text:     The cleaned input text
        risk_score:       0-100 ensemble risk score
        url_features:     Dict of URL features (optional)
    
    Returns:
        dict: {
            'triggered_signals': list of signal categories found,
            'reasons': list of human-readable reason strings,
            'severity_breakdown': count by severity level,
            'primary_reason': most important single reason,
            'scam_type': likely type of scam
        }
    """
    
    text_lower = cleaned_text.lower()
    
    # Get scam words from SHAP
    shap_scam_words = set()
    if shap_explanation and 'top_scam_words' in shap_explanation:
        for item in shap_explanation['top_scam_words']:
            shap_scam_words.add(item['word'].lower())
    
    # ── Check each signal category ────────────────────────
    triggered = []
    
    for category_name, category_data in SIGNAL_CATEGORIES.items():
        # Check if any keyword from this category appears
        # in EITHER the text OR the SHAP top words
        text_match = any(
            kw in text_lower 
            for kw in category_data['keywords']
        )
        shap_match = any(
            kw in shap_scam_words 
            for kw in category_data['keywords']
        )
        
        if text_match or shap_match:
            triggered.append({
                'category':  category_name,
                'reason':    category_data['reason'],
                'severity':  category_data['severity'],
                'advice':    category_data['advice']
            })
    
    # ── Add URL-based reasons ──────────────────────────────
    if url_features:
        if url_features.get('has_ip_address'):
            triggered.append({
                'category': 'ip_url',
                'reason':   'IP address used as URL — websites use domain names, not IP addresses',
                'severity': 'CRITICAL',
                'advice':   'Never visit websites with IP addresses in the URL'
            })
        if url_features.get('is_suspicious_tld'):
            triggered.append({
                'category': 'suspicious_tld',
                'reason':   'Suspicious domain extension detected (.xyz, .tk, .ml are common in scam sites)',
                'severity': 'HIGH',
                'advice':   'Legitimate organizations use .com, .in, .gov.in, .org domains'
            })
        if url_features.get('brand_in_subdomain_or_path'):
            triggered.append({
                'category': 'brand_impersonation_url',
                'reason':   'Brand name in URL path but not as official domain — classic impersonation',
                'severity': 'CRITICAL',
                'advice':   'Check that the DOMAIN (not path) matches the official website exactly'
            })
        if url_features.get('has_multiple_subdomains'):
            triggered.append({
                'category': 'multiple_subdomains',
                'reason':   'Excessive subdomains detected — used to make URLs look official',
                'severity': 'MEDIUM',
                'advice':   'Official sites rarely have more than 2 subdomain levels'
            })
    
    # ── Handle no signals case ────────────────────────────
    if not triggered:
        if risk_score > 50:
            triggered.append({
                'category': 'statistical_pattern',
                'reason':   'Statistical patterns similar to known scam messages detected',
                'severity': 'MEDIUM',
                'advice':   'Be cautious. Verify through official channels before taking action.'
            })
    
    # ── Determine primary reason ──────────────────────────
    severity_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
    triggered_sorted = sorted(
        triggered,
        key=lambda x: severity_order.get(x['severity'], 3)
    )
    
    primary_reason = triggered_sorted[0]['reason'] if triggered_sorted else \
                     "Pattern analysis indicates potential risk"
    
    # ── Determine likely scam type ────────────────────────
    scam_type = 'Unknown Scam'
    category_names = [t['category'] for t in triggered]
    
    if 'otp_request' in category_names:
        scam_type = 'OTP/Credential Theft'
    elif 'upi_fraud' in category_names:
        scam_type = 'UPI Payment Fraud'
    elif 'prize_lottery' in category_names:
        scam_type = 'Lottery/Prize Scam'
    elif 'financial_lure' in category_names and 'urgency' in category_names:
        scam_type = 'Financial Fraud'
    elif 'impersonation' in category_names:
        scam_type = 'Impersonation Scam'
    elif 'suspicious_url' in category_names or 'ip_url' in category_names:
        scam_type = 'Phishing Attack'
    elif 'personal_info_request' in category_names:
        scam_type = 'Personal Data Theft'
    elif 'account_threat' in category_names:
        scam_type = 'Account Takeover Attempt'
    
    # ── Severity breakdown ────────────────────────────────
    severity_breakdown = {'CRITICAL': 0, 'HIGH': 0, 'MEDIUM': 0, 'LOW': 0}
    for t in triggered:
        sev = t.get('severity', 'LOW')
        severity_breakdown[sev] = severity_breakdown.get(sev, 0) + 1
    
    return {
        'triggered_signals':  triggered_sorted,
        'reasons':            [t['reason'] for t in triggered_sorted],
        'severity_breakdown': severity_breakdown,
        'primary_reason':     primary_reason,
        'scam_type':          scam_type,
        'signal_count':       len(triggered)
    }


print("Reason generator ready!")
print(f"  Signal categories defined: {len(SIGNAL_CATEGORIES)}")
print()

# Quick test
test_text = "urgent your sbi account block verify otp immediately url_present"
dummy_shap = {
    'top_scam_words': [
        {'word': 'urgent', 'shap_value': 0.3, 'impact': 'scam'},
        {'word': 'otp', 'shap_value': 0.28, 'impact': 'scam'},
        {'word': 'block', 'shap_value': 0.21, 'impact': 'scam'},
    ],
    'top_legit_words': []
}
test_reasons = generate_reasons(dummy_shap, test_text, risk_score=88)

print("Test reason generation:")
print(f"  Scam type:      {test_reasons['scam_type']}")
print(f"  Signals found:  {test_reasons['signal_count']}")
print(f"  Primary reason: {test_reasons['primary_reason'][:60]}...")
print(f"  Severities:     {test_reasons['severity_breakdown']}")

Reason generator ready!
  Signal categories defined: 10

Test reason generation:
  Scam type:      OTP/Credential Theft
  Signals found:  7
  Primary reason: OTP/Password request detected — NO legitimate service asks y...
  Severities:     {'CRITICAL': 2, 'HIGH': 5, 'MEDIUM': 0, 'LOW': 0}


In [6]:
# ============================================================
# ADVICE GENERATOR
# ============================================================
# "What should I do now?" — This is what users REALLY need.
# Generic advice is useless. We give SPECIFIC advice
# based on the scam type and risk level.
# ============================================================

# ── Emergency resources ───────────────────────────────────
EMERGENCY_RESOURCES = {
    'cybercrime_portal': 'https://cybercrime.gov.in',
    'cybercrime_helpline': '1930',
    'rbi_helpline': '14440',
    'consumer_helpline': '1800-11-4000',
    'email': 'cybercrime.gov.in/Report.aspx'
}

# ── Scam-type specific advice ─────────────────────────────
SCAM_ADVICE = {
    'OTP/Credential Theft': {
        'immediate': [
            '🚫 NEVER share OTP with anyone — not even bank employees',
            '🚫 Do NOT call back any number mentioned in the message',
            '🔒 If you already shared OTP: call your bank immediately at their official number',
            '🔒 Block the sender\'s number',
        ],
        'preventive': [
            '✅ Enable SMS transaction alerts on your bank account',
            '✅ Set daily transaction limits on your banking app',
            '✅ Register on DND (Do Not Disturb) at trai.gov.in',
        ],
        'report_to': [
            f'📞 Cybercrime Helpline: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            f'🌐 Online: {EMERGENCY_RESOURCES["cybercrime_portal"]}',
            '📱 Forward scam SMS to 7726 (SPAM)',
        ]
    },
    'UPI Payment Fraud': {
        'immediate': [
            '🚫 Do NOT scan any QR code sent to you',
            '🚫 Do NOT click "Collect Request" from unknown UPI IDs',
            '🔒 Remember: Receiving money NEVER requires your UPI PIN',
            '🔒 If already transacted: Report in your UPI app immediately',
        ],
        'preventive': [
            '✅ Enable UPI transaction notifications',
            '✅ Set transaction limits in your UPI app',
            '✅ Block unknown UPI collection requests',
        ],
        'report_to': [
            f'📞 Cybercrime: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            '🏦 Your bank\'s fraud helpline (number on bank card)',
            f'🌐 {EMERGENCY_RESOURCES["cybercrime_portal"]}',
        ]
    },
    'Lottery/Prize Scam': {
        'immediate': [
            '🚫 Do NOT pay any "processing fee" or "tax" to claim prize',
            '🚫 Do NOT share personal documents',
            '🚫 Do NOT call the number provided',
            '❌ You CANNOT win a lottery you never entered',
        ],
        'preventive': [
            '✅ Block and report the number',
            '✅ Warn family members about this scam',
        ],
        'report_to': [
            f'📞 Cybercrime: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            f'🌐 {EMERGENCY_RESOURCES["cybercrime_portal"]}',
        ]
    },
    'Phishing Attack': {
        'immediate': [
            '🚫 Do NOT click any links in the message',
            '🚫 Do NOT enter credentials on any linked website',
            '🔒 If you clicked and entered details: change passwords immediately',
            '🔒 Enable 2-factor authentication on all accounts',
        ],
        'preventive': [
            '✅ Always check URL carefully before entering password',
            '✅ Look for HTTPS and the correct domain name',
            '✅ Use a password manager to avoid fake site logins',
            '✅ Install antivirus with web protection',
        ],
        'report_to': [
            f'📞 Cybercrime: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            '📧 Forward phishing emails to: report@phishing.gov.in',
            '🌐 PhishTank: phishtank.com/report',
        ]
    },
    'Impersonation Scam': {
        'immediate': [
            '🚫 Do NOT trust caller ID or SMS sender name',
            '🚫 Do NOT share any information over phone/SMS',
            '📞 Hang up and call the organization directly',
            '🔍 Verify using official website/app ONLY',
        ],
        'preventive': [
            '✅ Save official helpline numbers of your bank/services',
            '✅ Banks never ask for PIN/OTP over phone',
            '✅ Government agencies contact via official letters, not SMS',
        ],
        'report_to': [
            f'📞 Cybercrime: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            '🏦 Your bank fraud helpline',
            f'🌐 {EMERGENCY_RESOURCES["cybercrime_portal"]}',
        ]
    },
    'Personal Data Theft': {
        'immediate': [
            '🚫 Do NOT share Aadhaar, PAN, passport details via SMS/email',
            '🚫 Do NOT upload documents to links sent in messages',
            '🔒 If already shared: file a complaint immediately',
        ],
        'preventive': [
            '✅ Lock your Aadhaar biometrics at myaadhaar.uidai.gov.in',
            '✅ Check Aadhaar authentication history regularly',
            '✅ Never share documents with unverified parties',
        ],
        'report_to': [
            f'📞 UIDAI Helpline: 1947',
            f'📞 Cybercrime: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            f'🌐 {EMERGENCY_RESOURCES["cybercrime_portal"]}',
        ]
    },
    'Unknown Scam': {
        'immediate': [
            '🚫 Do NOT respond to the message',
            '🚫 Do NOT click any links or call any numbers',
            '🔒 Block the sender',
        ],
        'preventive': [
            '✅ Be skeptical of unsolicited messages',
            '✅ Verify independently before taking any action',
        ],
        'report_to': [
            f'📞 Cybercrime Helpline: {EMERGENCY_RESOURCES["cybercrime_helpline"]}',
            f'🌐 {EMERGENCY_RESOURCES["cybercrime_portal"]}',
        ]
    }
}

# ── Risk-level based opening statement ────────────────────
RISK_OPENING = {
    'CRITICAL': {
        'en': '🔴 CRITICAL RISK — This is almost certainly a scam. Do NOT engage.',
        'hi': '🔴 गंभीर खतरा — यह लगभग निश्चित रूप से एक घोटाला है। कोई कार्रवाई न करें।'
    },
    'HIGH': {
        'en': '🚨 HIGH RISK — This shows strong signs of being a scam. Be very careful.',
        'hi': '🚨 उच्च जोखिम — इसमें घोटाले के स्पष्ट संकेत हैं। बहुत सावधान रहें।'
    },
    'MEDIUM': {
        'en': '⚠️ MEDIUM RISK — This has some suspicious elements. Verify before acting.',
        'hi': '⚠️ मध्यम जोखिम — इसमें कुछ संदिग्ध तत्व हैं। कार्रवाई से पहले सत्यापित करें।'
    },
    'LOW': {
        'en': '✅ LOW RISK — This appears to be legitimate. Stay cautious anyway.',
        'hi': '✅ कम जोखिम — यह वैध प्रतीत होता है। फिर भी सतर्क रहें।'
    }
}


def generate_advice(scam_type, risk_level, risk_score):
    """
    Generate specific actionable advice based on scam type and risk.
    
    Args:
        scam_type:   String from generate_reasons() output
        risk_level:  LOW/MEDIUM/HIGH/CRITICAL
        risk_score:  0-100 numeric score
    
    Returns:
        dict: {
            'opening': risk-level statement,
            'immediate_actions': list of urgent steps,
            'preventive_measures': list of prevention tips,
            'report_to': list of reporting options,
            'summary': one-line summary
        }
    """
    
    # Get advice for this scam type (fall back to Unknown)
    advice_data = SCAM_ADVICE.get(scam_type, SCAM_ADVICE['Unknown Scam'])
    
    # Get opening statement
    opening = RISK_OPENING.get(risk_level, RISK_OPENING['MEDIUM'])['en']
    
    # Build summary
    if risk_score >= 76:
        summary = f"⚠️ SCAM ALERT: {scam_type} detected with {risk_score:.0f}% confidence"
    elif risk_score >= 51:
        summary = f"🔍 SUSPICIOUS: Possible {scam_type} — verify before acting"
    elif risk_score >= 26:
        summary = f"⚡ CAUTION: Some suspicious patterns detected — stay alert"
    else:
        summary = f"✅ SAFE: Message appears legitimate (score: {risk_score:.0f}/100)"
    
    return {
        'opening':             opening,
        'immediate_actions':   advice_data['immediate'],
        'preventive_measures': advice_data['preventive'],
        'report_to':           advice_data['report_to'],
        'summary':             summary,
        'emergency_number':    EMERGENCY_RESOURCES['cybercrime_helpline'],
        'portal':              EMERGENCY_RESOURCES['cybercrime_portal']
    }


# ── Test advice generator ─────────────────────────────────
print("Testing advice generator:")
print()

test_cases = [
    ('OTP/Credential Theft', 'CRITICAL', 94),
    ('Lottery/Prize Scam', 'HIGH', 82),
    ('Phishing Attack', 'HIGH', 71),
    ('Unknown Scam', 'LOW', 15),
]

for scam_type, risk_level, score in test_cases:
    advice = generate_advice(scam_type, risk_level, score)
    print(f"  Scam type: {scam_type}")
    print(f"  {advice['summary']}")
    print(f"  Opening: {advice['opening']}")
    print(f"  First action: {advice['immediate_actions'][0]}")
    print()

Testing advice generator:

  Scam type: OTP/Credential Theft
  ⚠️ SCAM ALERT: OTP/Credential Theft detected with 94% confidence
  Opening: 🔴 CRITICAL RISK — This is almost certainly a scam. Do NOT engage.
  First action: 🚫 NEVER share OTP with anyone — not even bank employees

  Scam type: Lottery/Prize Scam
  ⚠️ SCAM ALERT: Lottery/Prize Scam detected with 82% confidence
  Opening: 🚨 HIGH RISK — This shows strong signs of being a scam. Be very careful.
  First action: 🚫 Do NOT pay any "processing fee" or "tax" to claim prize

  Scam type: Phishing Attack
  🔍 SUSPICIOUS: Possible Phishing Attack — verify before acting
  Opening: 🚨 HIGH RISK — This shows strong signs of being a scam. Be very careful.
  First action: 🚫 Do NOT click any links in the message

  Scam type: Unknown Scam
  ✅ SAFE: Message appears legitimate (score: 15/100)
  Opening: ✅ LOW RISK — This appears to be legitimate. Stay cautious anyway.
  First action: 🚫 Do NOT respond to the message



In [7]:
# ============================================================
# MULTILINGUAL SUPPORT — Hindi + English
# ============================================================
# India has 600M+ Hindi speakers.
# Our tool must work for them too.
#
# STRATEGY:
# 1. Detect input language (Hindi or English)
# 2. If Hindi input: translate to English for model
# 3. Run analysis in English
# 4. Translate explanation output back to Hindi
#
# We use deep-translator (replaced googletrans)
# which is stable and doesn't conflict with our stack.
# ============================================================

def detect_language(text):
    """
    Detect the language of input text.
    
    Returns:
        str: 'hi' for Hindi, 'en' for English, 'unknown' otherwise
    """
    try:
        # langdetect needs at least a few words to work reliably
        if len(text.strip()) < 10:
            return 'en'  # Default to English for short texts
        
        detected = detect(text)
        
        # langdetect returns 'hi' for Hindi, 'en' for English
        if detected in ['hi', 'en']:
            return detected
        
        # Check for Devanagari script (Hindi characters)
        # Unicode range for Devanagari: U+0900 to U+097F
        hindi_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
        if hindi_chars > 3:
            return 'hi'
        
        return 'en'  # Default to English
        
    except Exception:
        return 'en'


def translate_text(text, source_lang, target_lang):
    """
    Translate text between Hindi and English.
    
    Args:
        text:        Text to translate
        source_lang: 'en' or 'hi'
        target_lang: 'en' or 'hi'
    
    Returns:
        str: Translated text, or original if translation fails
    """
    
    # No translation needed
    if source_lang == target_lang:
        return text
    
    # Empty text
    if not text or len(text.strip()) < 3:
        return text
    
    try:
        translated = GoogleTranslator(
            source=source_lang,
            target=target_lang
        ).translate(text)
        
        return translated if translated else text
        
    except Exception as e:
        # If translation fails, return original
        # We never want translation failure to break the analysis
        print(f"Translation warning: {e}")
        return text


def get_hindi_explanation(explanation_dict):
    """
    Translate key parts of the explanation to Hindi.
    
    Args:
        explanation_dict: Full explanation from get_full_explanation()
    
    Returns:
        dict: Hindi versions of key fields
    """
    
    hindi_explanation = {}
    
    # Translate the summary
    summary_en = explanation_dict.get('advice', {}).get('summary', '')
    hindi_explanation['summary_hi'] = translate_text(summary_en, 'en', 'hi')
    
    # Translate primary reason
    primary_reason_en = explanation_dict.get('reasons', {}).get('primary_reason', '')
    hindi_explanation['primary_reason_hi'] = translate_text(
        primary_reason_en, 'en', 'hi'
    )
    
    # Translate opening statement
    risk_level = explanation_dict.get('risk_level', 'MEDIUM')
    hindi_explanation['opening_hi'] = RISK_OPENING.get(
        risk_level, RISK_OPENING['MEDIUM']
    )['hi']
    
    # Translate first immediate action
    immediate_actions = explanation_dict.get('advice', {}).get('immediate_actions', [])
    if immediate_actions:
        hindi_explanation['first_action_hi'] = translate_text(
            immediate_actions[0], 'en', 'hi'
        )
    
    return hindi_explanation


# ── Test multilingual ─────────────────────────────────────
print("Testing multilingual support:")
print()

# Test language detection
test_texts = [
    ("Your SBI account will be blocked! Verify OTP now.", 'en'),
    ("आपका SBI खाता ब्लॉक हो जाएगा! अभी OTP सत्यापित करें।", 'hi'),
    ("Aapka Paytm account suspend ho jayega KYC update karein", 'en'),
    ("मेरा नाम राहुल है और मैं दिल्ली में रहता हूँ", 'hi'),
]

for text, expected in test_texts:
    detected = detect_language(text)
    status = "✅" if detected == expected else "⚠️"
    print(f"  {status} Detected: '{detected}' | Expected: '{expected}'")
    print(f"     Text: '{text[:50]}...'")
    print()

# Test translation
print("Testing translation:")
test_en = "This message is a scam. Do not share your OTP with anyone."
test_hi = translate_text(test_en, 'en', 'hi')
print(f"  English: {test_en}")
print(f"  Hindi:   {test_hi}")
print()

# Test Hindi to English (for model input)
test_hindi_input = "आपका बैंक खाता बंद हो जाएगा। अभी OTP साझा करें।"
test_translated  = translate_text(test_hindi_input, 'hi', 'en')
print(f"  Hindi input:       {test_hindi_input}")
print(f"  Translated to EN:  {test_translated}")

Testing multilingual support:

  ✅ Detected: 'en' | Expected: 'en'
     Text: 'Your SBI account will be blocked! Verify OTP now....'

  ✅ Detected: 'hi' | Expected: 'hi'
     Text: 'आपका SBI खाता ब्लॉक हो जाएगा! अभी OTP सत्यापित करे...'

  ✅ Detected: 'en' | Expected: 'en'
     Text: 'Aapka Paytm account suspend ho jayega KYC update k...'

  ✅ Detected: 'hi' | Expected: 'hi'
     Text: 'मेरा नाम राहुल है और मैं दिल्ली में रहता हूँ...'

Testing translation:
  English: This message is a scam. Do not share your OTP with anyone.
  Hindi:   यह संदेश एक घोटाला है. अपना ओटीपी किसी के साथ साझा न करें.

  Hindi input:       आपका बैंक खाता बंद हो जाएगा। अभी OTP साझा करें।
  Translated to EN:  Your bank account will be closed. Share OTP now.


In [8]:
# ============================================================
# MASTER EXPLANATION FUNCTION
# ============================================================
# This is the SINGLE function our FastAPI backend will call.
# It orchestrates ALL explanation components:
#   1. Detect language
#   2. Translate if needed
#   3. Get SHAP explanation
#   4. Generate reasons
#   5. Generate advice
#   6. Add Hindi translation
#   7. Return complete package
# ============================================================

def get_full_explanation(
    input_text,
    tfidf,
    lr_model,
    lr_explainer,
    feature_names,
    risk_score,
    risk_level,
    url_features=None,
    preferred_language='en'
):
    """
    Master function: generates complete explanation for any input.
    
    Args:
        input_text:         Raw user input (any language)
        tfidf:              TF-IDF vectorizer
        lr_model:           Logistic Regression model
        lr_explainer:       SHAP LinearExplainer
        feature_names:      TF-IDF feature names
        risk_score:         0-100 ensemble risk score
        risk_level:         LOW/MEDIUM/HIGH/CRITICAL
        url_features:       Dict of URL features (optional)
        preferred_language: 'en' or 'hi' for output
    
    Returns:
        dict: Complete explanation package for frontend
    """
    
    # ── Step 1: Language Detection ────────────────────────
    input_lang = detect_language(input_text)
    
    # ── Step 2: Translate to English for model ────────────
    if input_lang == 'hi':
        english_text = translate_text(input_text, 'hi', 'en')
    else:
        english_text = input_text
    
    # ── Step 3: Basic text cleaning for analysis ──────────
    cleaned = english_text.lower()
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    # ── Step 4: SHAP Explanation ──────────────────────────
    try:
        shap_result = get_text_shap_explanation(
            cleaned, tfidf, lr_model, lr_explainer, feature_names
        )
    except Exception as e:
        shap_result = {
            'prediction': 1 if risk_score > 50 else 0,
            'probability': risk_score / 100,
            'top_scam_words': [],
            'top_legit_words': [],
            'all_contributions': {}
        }
    
    # ── Step 5: Generate Reasons ──────────────────────────
    reasons_result = generate_reasons(
        shap_result,
        cleaned,
        risk_score,
        url_features
    )
    
    # ── Step 6: Generate Advice ───────────────────────────
    advice_result = generate_advice(
        reasons_result['scam_type'],
        risk_level,
        risk_score
    )
    
    # ── Step 7: Build top contributing words list ─────────
    top_words = []
    for item in shap_result.get('top_scam_words', [])[:7]:
        top_words.append({
            'word':   item['word'],
            'impact': 'scam',
            'score':  round(item['shap_value'], 4)
        })
    for item in shap_result.get('top_legit_words', [])[:3]:
        top_words.append({
            'word':   item['word'],
            'impact': 'legitimate',
            'score':  round(abs(item['shap_value']), 4)
        })
    
    # ── Step 8: Compile final output ──────────────────────
    explanation = {
        # Input info
        'input_language':   input_lang,
        'was_translated':   (input_lang == 'hi'),
        
        # Risk info
        'risk_score':   risk_score,
        'risk_level':   risk_level,
        
        # What was found
        'scam_type':        reasons_result['scam_type'],
        'signal_count':     reasons_result['signal_count'],
        'primary_reason':   reasons_result['primary_reason'],
        'all_reasons':      reasons_result['reasons'][:5],  # Top 5
        'triggered_signals': [
            {
                'category': s['category'],
                'reason':   s['reason'],
                'severity': s['severity']
            }
            for s in reasons_result['triggered_signals'][:5]
        ],
        
        # SHAP word highlights
        'top_words': top_words,
        
        # Advice
        'advice': advice_result,
        
        # Summary
        'summary': advice_result['summary'],
        'opening': advice_result['opening'],
    }
    
    # ── Step 9: Add Hindi translation if requested ────────
    if preferred_language == 'hi' or input_lang == 'hi':
        hindi_parts = get_hindi_explanation(explanation)
        explanation['hindi'] = hindi_parts
    
    return explanation


# ── FULL INTEGRATION TEST ─────────────────────────────────
print("=" * 60)
print("  FULL INTEGRATION TEST — Complete Explanation Pipeline")
print("=" * 60)
print()

integration_tests = [
    {
        'text': "URGENT: Your SBI account blocked! Share OTP 847291 NOW to restore. Click: http://sbi-verify.xyz",
        'risk_score': 92,
        'risk_level': 'CRITICAL',
        'lang': 'en'
    },
    {
        'text': "आपका Paytm खाता बंद हो जाएगा! KYC अपडेट करें: http://paytm-kyc.xyz",
        'risk_score': 87,
        'risk_level': 'CRITICAL',
        'lang': 'hi'
    },
    {
        'text': "Your Amazon order #12345 has been shipped. Delivery by Jan 20.",
        'risk_score': 8,
        'risk_level': 'LOW',
        'lang': 'en'
    },
]

for test in integration_tests:
    print(f"INPUT TEXT: '{test['text'][:70]}...'")
    print(f"RISK: {test['risk_level']} ({test['risk_score']}/100)")
    print()
    
    result = get_full_explanation(
        input_text=test['text'],
        tfidf=tfidf,
        lr_model=lr_model,
        lr_explainer=lr_explainer,
        feature_names=feature_names,
        risk_score=test['risk_score'],
        risk_level=test['risk_level'],
        preferred_language=test['lang']
    )
    
    print(f"  📋 Summary:        {result['summary']}")
    print(f"  🎯 Scam Type:      {result['scam_type']}")
    print(f"  🔍 Signals Found:  {result['signal_count']}")
    print(f"  ⚠️  Primary Reason: {result['primary_reason'][:60]}...")
    
    if result['top_words']:
        scam_words = [w['word'] for w in result['top_words'] 
                      if w['impact'] == 'scam'][:5]
        print(f"  🚩 Key Words:      {', '.join(scam_words)}")
    
    if result.get('hindi'):
        print(f"  🇮🇳 Hindi Summary: {result['hindi'].get('summary_hi', 'N/A')[:60]}")
    
    print(f"  📞 Report to:      {result['advice']['emergency_number']}")
    print()
    print("-" * 60)
    print()

  FULL INTEGRATION TEST — Complete Explanation Pipeline

INPUT TEXT: 'URGENT: Your SBI account blocked! Share OTP 847291 NOW to restore. Cli...'
RISK: CRITICAL (92/100)

  📋 Summary:        ⚠️ SCAM ALERT: OTP/Credential Theft detected with 92% confidence
  🎯 Scam Type:      OTP/Credential Theft
  🔍 Signals Found:  7
  ⚠️  Primary Reason: OTP/Password request detected — NO legitimate service asks y...
  🚩 Key Words:      urgent, account, verify, otp, share otp
  📞 Report to:      1930

------------------------------------------------------------

INPUT TEXT: 'आपका Paytm खाता बंद हो जाएगा! KYC अपडेट करें: http://paytm-kyc.xyz...'
RISK: CRITICAL (87/100)

  📋 Summary:        ⚠️ SCAM ALERT: UPI Payment Fraud detected with 87% confidence
  🎯 Scam Type:      UPI Payment Fraud
  🔍 Signals Found:  6
  ⚠️  Primary Reason: Personal information request detected — asking for sensitive...
  🚩 Key Words:      account, kyc, update, paytm, paytm account
  🇮🇳 Hindi Summary: ⚠️ घोटाला चेतावनी: 87% विश्व

In [9]:
# ============================================================
# SAVE ALL EXPLAINABILITY COMPONENTS TO BACKEND
# ============================================================
# The backend needs access to all these functions.
# We save them as proper Python modules.
# ============================================================

import pickle
import json

print("Saving explainability configuration...")

# Save signal categories config
signal_config = {
    'categories': {
        name: {
            'keywords': data['keywords'],
            'reason': data['reason'],
            'severity': data['severity'],
            'advice': data['advice']
        }
        for name, data in SIGNAL_CATEGORIES.items()
    }
}

with open(os.path.join(MODELS_DIR, 'signal_categories.json'), 'w', 
          encoding='utf-8') as f:
    json.dump(signal_config, f, indent=2, ensure_ascii=False)
print(f"  [OK] signal_categories.json")

# Save scam advice config
with open(os.path.join(MODELS_DIR, 'scam_advice.json'), 'w',
          encoding='utf-8') as f:
    json.dump(SCAM_ADVICE, f, indent=2, ensure_ascii=False)
print(f"  [OK] scam_advice.json")

# Save risk opening statements
with open(os.path.join(MODELS_DIR, 'risk_openings.json'), 'w',
          encoding='utf-8') as f:
    json.dump(RISK_OPENING, f, indent=2, ensure_ascii=False)
print(f"  [OK] risk_openings.json")

# Save SHAP explainer
with open(os.path.join(MODELS_DIR, 'shap_lr_explainer.pkl'), 'wb') as f:
    pickle.dump(lr_explainer, f)
print(f"  [OK] shap_lr_explainer.pkl")

# Save emergency resources
with open(os.path.join(MODELS_DIR, 'emergency_resources.json'), 'w') as f:
    json.dump(EMERGENCY_RESOURCES, f, indent=2)
print(f"  [OK] emergency_resources.json")

print()
print("=" * 60)
print("  PHASE 3 COMPLETE!")
print("=" * 60)
print()
print("Components built and saved:")
print("  ✅ SHAP LinearExplainer (LR model)")
print("  ✅ SHAP visualizations (summary + individual)")
print("  ✅ Word-level explanation function")
print("  ✅ Reason generator (10 signal categories)")
print("  ✅ Advice generator (7 scam types)")
print("  ✅ Multilingual support (English + Hindi)")
print("  ✅ Master explanation orchestrator")
print()
print("Files saved to backend/saved_models/:")
files = [
    'shap_lr_explainer.pkl',
    'signal_categories.json',
    'scam_advice.json',
    'risk_openings.json',
    'emergency_resources.json'
]
for f in files:
    path = os.path.join(MODELS_DIR, f)
    size = os.path.getsize(path) / 1024  # KB
    print(f"  [{size:.1f} KB] {f}")
print()
print("NEXT: Phase 4 — FastAPI Backend Development")

Saving explainability configuration...
  [OK] signal_categories.json
  [OK] scam_advice.json
  [OK] risk_openings.json
  [OK] shap_lr_explainer.pkl
  [OK] emergency_resources.json

  PHASE 3 COMPLETE!

Components built and saved:
  ✅ SHAP LinearExplainer (LR model)
  ✅ SHAP visualizations (summary + individual)
  ✅ Word-level explanation function
  ✅ Reason generator (10 signal categories)
  ✅ Advice generator (7 scam types)
  ✅ Multilingual support (English + Hindi)
  ✅ Master explanation orchestrator

Files saved to backend/saved_models/:
  [270.0 KB] shap_lr_explainer.pkl
  [5.0 KB] signal_categories.json
  [4.2 KB] scam_advice.json
  [1.2 KB] risk_openings.json
  [0.2 KB] emergency_resources.json

NEXT: Phase 4 — FastAPI Backend Development
